# Notebook 2 — Data Processing and Feature Engineering

This notebook merges all data sources from GEE and engineers features for the flood prediction model.

## Inputs (from Notebook 1)
- dhemaji_flood_labels_multiyear.csv
- local_rainfall_multiyear.csv
- upstream_rainfall_multiyear.csv
- brahmaputra_runoff.csv
- dhemaji_rivers.shp (+ .dbf, .shx, .prj)
- static_features.csv (elevation, slope, tree_cover)

## Output
- dhemaji_flood_FINAL.csv — Final cleaned and merged dataset

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union

# Allow geopandas to recreate missing .shx file
os.environ["SHAPE_RESTORE_SHX"] = "YES"

## 2. Load and Clean SAR Flood Labels

In [ ]:
df_sar = pd.read_csv("../data/raw/dhemaji_flood_labels_multiyear.csv")

# Extract lat/lon from .geo column
df_sar[["longitude","latitude"]] = df_sar[".geo"].apply(
    lambda x: pd.Series(json.loads(x)["coordinates"])
)

# Fix date format
df_sar["date"] = pd.to_datetime(df_sar["date"], format="%d-%m-%Y")

# Remove no-coverage cells
df_sar = df_sar[df_sar["flood_label"] != -1]

# Add 500m grid keys
df_sar["grid_lat"] = (df_sar["latitude"] // 0.0045)
df_sar["grid_lon"] = (df_sar["longitude"] // 0.0045)

df_sar = df_sar[[
    "grid_lat","grid_lon",
    "latitude","longitude",
    "date","flood_label"
]]

print("SAR shape:", df_sar.shape)
print("Unique dates:", df_sar["date"].nunique())
print("Flood balance:")
print(df_sar["flood_label"].value_counts(normalize=True).round(3))

## 3. Load and Clean Local Rainfall

In [ ]:
df_local = pd.read_csv("../data/raw/local_rainfall_multiyear.csv")

# Extract lat/lon
df_local[["longitude","latitude"]] = df_local[".geo"].apply(
    lambda x: pd.Series(json.loads(x)["coordinates"])
)

# Fix date
df_local["date"] = pd.to_datetime(df_local["date"], format="%d-%m-%Y")

# Add grid keys
df_local["grid_lat"] = (df_local["latitude"] // 0.0045)
df_local["grid_lon"] = (df_local["longitude"] // 0.0045)

# Rename precipitation to rainfall_mm if needed
if "precipitation" in df_local.columns:
    df_local = df_local.rename(columns={"precipitation":"rainfall_mm"})

df_local = df_local[[
    "grid_lat","grid_lon","date","rainfall_mm"
]]

print("Local rainfall shape:", df_local.shape)

## 4. Merge SAR Labels with Local Rainfall

In [ ]:
df_merged = df_sar.merge(
    df_local,
    on=["grid_lat","grid_lon","date"],
    how="inner"
)

print("After SAR + rainfall merge:", df_merged.shape)

## 5. Engineer Rainfall Features

Create cumulative, lagged, and anomaly features for rainfall.

In [ ]:
# Sort for rolling windows
df_merged = df_merged.sort_values(
    ["grid_lat","grid_lon","date"]
).reset_index(drop=True)

# Rolling cumulative rainfall
df_merged["rain_3day"] = df_merged.groupby(
    ["grid_lat","grid_lon"]
)["rainfall_mm"].transform(
    lambda x: x.rolling(window=3, min_periods=1).sum()
)

df_merged["rain_5day"] = df_merged.groupby(
    ["grid_lat","grid_lon"]
)["rainfall_mm"].transform(
    lambda x: x.rolling(window=5, min_periods=1).sum()
)

# Rainfall anomaly
mean_rain = df_merged.groupby(
    ["grid_lat","grid_lon"]
)["rainfall_mm"].transform("mean")
df_merged["rain_anomaly"] = df_merged["rainfall_mm"] - mean_rain

df_merged = df_merged.fillna(0)

print("Rainfall features added")
print("New columns:", ["rain_3day","rain_5day","rain_anomaly"])

## 6. Add Upstream Rainfall (Optional Reference Features)

In [ ]:
df_upstream = pd.read_csv("../data/raw/upstream_rainfall_multiyear.csv")

df_upstream["date"] = pd.to_datetime(df_upstream["date"], format="%d-%m-%Y")
df_upstream = df_upstream[["date","upstream_rainfall"]]
df_upstream = df_upstream.sort_values("date").reset_index(drop=True)

# Merge on date — upstream rainfall is same for all cells on a given date
df_merged = df_merged.merge(
    df_upstream, on="date", how="left"
).fillna(0)

print("After upstream merge:", df_merged.shape)

## 7. Add ERA5 Runoff Features

In [ ]:
df_runoff = pd.read_csv("../data/raw/brahmaputra_runoff.csv")

df_runoff["date"] = pd.to_datetime(df_runoff["date"], format="%d-%m-%Y")
df_runoff = df_runoff[["date","runoff_sum"]]
df_runoff = df_runoff.sort_values("date").reset_index(drop=True)

# Runoff anomaly
df_runoff["runoff_anomaly"] = (
    df_runoff["runoff_sum"] - df_runoff["runoff_sum"].mean()
)

df_runoff = df_runoff.fillna(0)

# Merge on date
df_merged = df_merged.merge(
    df_runoff, on="date", how="left"
).fillna(0)

print("After runoff merge:", df_merged.shape)

## 8. Compute Distance to Major River

Single most important spatial feature. Computed from HydroSHEDS river network filtered to major rivers (order <= 4).

In [ ]:
# Load river shapefile
rivers_gdf = gpd.read_file("../data/raw/dhemaji_rivers.shp")
rivers_gdf = rivers_gdf.to_crs("EPSG:4326")

print("Total river segments:", len(rivers_gdf))
print("River order distribution:")
print(rivers_gdf["RIV_ORD"].value_counts())

# Filter to major rivers (Brahmaputra and main tributaries)
major_rivers = rivers_gdf[rivers_gdf["RIV_ORD"] <= 4]
river_union = unary_union(major_rivers.geometry)
print("Major rivers used:", len(major_rivers))

In [ ]:
# Calculate distance per unique grid cell (much faster than per row)
unique_cells = df_merged[
    ["grid_lat","grid_lon","latitude","longitude"]
].drop_duplicates()

print("Computing distance for", len(unique_cells), "cells...")

unique_cells["dist_to_major_river"] = unique_cells.apply(
    lambda row: Point(
        row["longitude"], row["latitude"]
    ).distance(river_union) * 111000,  # convert degrees to meters
    axis=1
)

# Merge distance back to main dataset
df_merged = df_merged.merge(
    unique_cells[["grid_lat","grid_lon","dist_to_major_river"]],
    on=["grid_lat","grid_lon"],
    how="left"
)

print("Distance to river added")
print(df_merged["dist_to_major_river"].describe())

## 9. Add Static Features (Elevation, Slope, Tree Cover)

In [ ]:
static_features = pd.read_csv("../data/raw/static_features.csv")

df_final = df_merged.merge(
    static_features[[
        "grid_lat","grid_lon",
        "elevation","slope","tree_cover"
    ]],
    on=["grid_lat","grid_lon"],
    how="left"
)

# Fix tree_cover missing values using forward/backward fill per cell
df_final["tree_cover"] = df_final.groupby(
    ["grid_lat","grid_lon"]
)["tree_cover"].transform(
    lambda x: x.ffill().bfill()
)

# Fill remaining with median
df_final["tree_cover"] = df_final["tree_cover"].fillna(
    df_final["tree_cover"].median()
)

print("Final shape:", df_final.shape)
print("Missing values:")
print(df_final.isna().sum())

## 10. Add Year Column for Train-Test Split

In [ ]:
df_final["year"] = df_final["date"].dt.year

print("Years in dataset:", sorted(df_final["year"].unique()))
print("Rows per year:")
print(df_final["year"].value_counts().sort_index())

## 11. Validate Dataset

In [ ]:
# Check that labels are dynamic
variation = df_final.groupby(
    ["grid_lat","grid_lon"]
)["flood_label"].nunique()

print("Cells with changing labels:", (variation > 1).sum())
print("Cells with static labels:", (variation == 1).sum())

# Flood rate per year
print("\nFlood rate per year:")
print(df_final.groupby("year")["flood_label"].mean().round(3))

# Class balance
print("\nClass balance:")
print(df_final["flood_label"].value_counts(normalize=True).round(3))

## 12. Save Final Dataset

In [ ]:
# Keep only relevant columns
final_cols = [
    # Identity
    "grid_lat","grid_lon",
    "latitude","longitude",
    "date","year",
    
    # Label
    "flood_label",
    
    # Spatial features
    "dist_to_major_river",
    "elevation","slope","tree_cover",
    
    # Rainfall features
    "rainfall_mm",
    "rain_3day","rain_5day",
    "rain_anomaly",
    
    # Upstream / runoff
    "upstream_rainfall",
    "runoff_sum","runoff_anomaly"
]

available = [c for c in final_cols if c in df_final.columns]
df_save = df_final[available].copy()

df_save.to_csv("../data/processed/dhemaji_flood_FINAL.csv", index=False)

print("Final dataset saved:")
print("  Path:", "../data/processed/dhemaji_flood_FINAL.csv")
print("  Shape:", df_save.shape)
print("  Columns:", df_save.columns.tolist())